In [3]:
! pip install -U spacy
! python -m spacy download en_core_web_md

     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
      --------------------------------------- 0.5/33.5 MB 4.2 MB/s eta 0:00:08
     - -------------------------------------- 1.3/33.5 MB 4.0 MB/s eta 0:00:09
     -- ------------------------------------- 1.8/33.5 MB 3.0 MB/s eta 0:00:11
     --- ------------------------------------ 2.9/33.5 MB 3.7 MB/s eta 0:00:09
     --- ------------------------------------ 2.9/33.5 MB 3.7 MB/s eta 0:00:09
     --- ------------------------------------ 2.9/33.5 MB 3.7 MB/s eta 0:00:09
     --- ------------------------------------ 2.9/33.5 MB 3.7 MB/s eta 0:00:09
     --- ------------------------------------ 2.9/33.5 MB 3.7 MB/s eta 0:00:09
     --- ------------------------------------ 2.9/33.5 MB 3.7 MB/s eta 0:00:09
     ---- ----------------------------------- 3.4/33.5 MB 1.6 MB/s eta 0:00:20
     ---- ----------------------------------- 3.9/33.5 MB 1.7 MB/s

Phân tích câu và Trực quan hóa 

In [7]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_md")
text = "The quick brown fox jumps over the lazy dog."
doc = nlp(text)

In [10]:
option = {"compact": True, "bg": "#09a3d5", "color": "white", "font": "Source Sans Pro"}

In [11]:
displacy.serve(doc , style="dep", options=option)


Using the 'dep' visualizer
Serving on http://0.0.0.0:5000 ...

Shutting down server on port 5000.


• Từ nào là gốc (ROOT) của câu?
    -jumps
• jumps có những từ phụ thuộc (dependent) nào? Các quan hệ đó
là gì?
    - fox, over
• fox là head của những từ nào?
    - browen, quich , the

Phần 3: Truy cập các thành phần trong cây phụ thuộc

In [15]:
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)
print(f"{'TEXT':<12}{'DEP':<10}{'HEAD TEXT':<12}{'HEAD POS':<8}{'CHILDREN'}")
print("-"*70)

TEXT        DEP       HEAD TEXT   HEAD POSCHILDREN
----------------------------------------------------------------------


In [18]:
for token in doc:
    children = [child for child in token.children]
    print(f"{token.text:<12}{token.dep_:<10}{token.head.text:<12}{token.head.pos_:<8}{[child for child in token.children]}")

Apple       nsubj     looking     VERB    []
is          aux       looking     VERB    []
looking     ROOT      looking     VERB    [Apple, is, at]
at          prep      looking     VERB    [buying]
buying      pcomp     at          ADP     [startup]
U.K.        compound  startup     NOUN    []
startup     dobj      buying      VERB    [U.K., for]
for         prep      startup     NOUN    [billion]
$           quantmod  billion     NUM     []
1           compound  billion     NUM     []
billion     pobj      for         ADP     [$, 1]


Phần 4: Duyệt cây để trích xuất thông tin 

In [19]:
text = "The cat chased the mouse and the dog watched them"
doc = nlp(text)
for token in doc:
    if token.pos_ == "VERB":
        verb = token.text
        subject = ""
        obj = ""
        for child in token.children:
            if child.dep_ == "nsubj":
                subject = child.text
            elif child.dep_ == "dobj":
                obj = child.text
if subject and obj:
        print(f'Founded triplet: ({subject}, {verb}, {obj})')

Founded triplet: (dog, watched, them)


In [20]:
text = "The big, fluffy white cat is sleeping on the warm mat"
doc = nlp(text)
for token in doc:
    if token.pos_ == "NOUN":
        adjectives = [child.text for child in token.children if child.dep_ == "amod"]
        if adjectives:
            print(f'Noun: {token.text} được bổ nghĩa bởi Adjectives: {adjectives}')

Noun: cat được bổ nghĩa bởi Adjectives: ['big', 'fluffy', 'white']
Noun: mat được bổ nghĩa bởi Adjectives: ['warm']


In [ ]:
Phần 5

Bài 1: Tìm động từ chính của câu

In [22]:
def find_main_verb(doc):
    for token in doc:
        if token.dep_ == "ROOT" and token.pos_ == "VERB":
            return token.text
    return None

In [23]:
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)
print("Main verb:", find_main_verb(doc))

Main verb: looking


In [31]:
def extract_noun_chunks_custom(doc):
    noun_chunks = []
    for token in doc:
        if token.pos_ in ("NOUN", "PROPN", "PRON"):
            chunk_tokens = []
            for child in token.children:
                if child.dep_ in ("det", "amod", "compound", "nummod", "nmod"):
                    chunk_tokens.append(child)
            chunk_tokens.append(token)
            
            chunk_tokens.sort(key=lambda t: t.i)   
            chunk_text = " ".join([t.text for t in chunk_tokens])
            noun_chunks.append((chunk_text, token))
    
    seen = set()
    unique_chunks = []
    for chunk_text, noun_token in noun_chunks:
        if chunk_text not in seen:
            seen.add(chunk_text)
            unique_chunks.append(chunk_text)
    
    return unique_chunks

In [28]:
def get_path_to_root(token):
    path = []
    while token.head != token:
        path.append(token.text)
        token = token.head
    path.append(token.text)  
    return " -> ".join(path[::-1])  

Bài 3: Tìm đường đi ngắn nhất trong cây


In [30]:
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)
custom_chunks = extract_noun_chunks_custom(doc)
print("Custom Noun Chunks:")
for chunk in custom_chunks:
    print(f"- {chunk}")
  



Custom Noun Chunks:
- Apple
- U.K.
- U.K. startup
